# 6. 신용 데이터 전처리 과제

Creditability를 예측 대상으로 분리하고, 연속형 변수와 정수로 기록된 범주형 변수에 다른 전처리를 적용합니다. 원본 CSV를 읽으려면 네트워크 연결이 필요합니다.

각 문제는 독립적으로 실행할 수 있습니다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data = pd.read_csv('https://raw.githubusercontent.com/DA4BAM/dataset/master/credit_NA.csv').drop(
    columns=['Purpose', 'MostValuable', 'NoCredits', 'Occupation', 'Telephone']
)

## 문제 1. 분할과 결측 탐색

Creditability를 분리하고 층화 분할한 뒤 훈련 세트의 열별 결측 개수를 출력하세요.

In [ ]:
eligible = data.dropna(subset=['Creditability'])
X = eligible.drop(columns='Creditability')
y = eligible['Creditability']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.isna().sum().sort_values(ascending=False))

**결과 해석**

정답을 알 수 있는 행으로 층화 분할하고, 결측 처리 방법은 훈련 세트를 바탕으로 정합니다.

## 문제 2. 결측 처리, 인코딩, 스케일링

Age, CreditAmount, CreditDuration에는 중앙값 대치와 표준화를 적용하세요. 나머지 특성에는 최빈값 대치와 원핫 인코딩을 적용하고 훈련과 테스트 결과의 열 수를 비교하세요. 이 문제에서 분할도 수행하세요.

In [ ]:
eligible = data.dropna(subset=['Creditability'])
X = eligible.drop(columns='Creditability')
y = eligible['Creditability']
X_train, X_test, _, _ = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
numeric = ['Age', 'CreditAmount', 'CreditDuration']
categorical = X_train.columns.difference(numeric).tolist()
preprocess = ColumnTransformer([
    ('numeric', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
    ('categorical', make_pipeline(SimpleImputer(strategy='most_frequent'),
                                 OneHotEncoder(handle_unknown='ignore')), categorical),
])
train_ready = preprocess.fit_transform(X_train)
test_ready = preprocess.transform(X_test)
print(train_ready.shape, test_ready.shape)

**결과 해석**

훈련에서 인코딩 열을 정하면 테스트에 새로운 범주가 있어도 변환 후 열 수가 유지됩니다.